# stack-vs-cat — ex1: pick stack or cat from the target shape

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `stack-vs-cat`. Running the final beacon cell reports progress against the `PyTorch: stack vs cat` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: stack vs cat` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`stack-vs-cat`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "stack-vs-cat"
DD_SUBTOPIC = "PyTorch: stack vs cat"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `stack` vs `cat` — quick refresher

- `t.cat(tensors, dim=k)` **concatenates** along an existing axis. Output rank = input rank. All inputs must agree on every axis except `k`.
- `t.stack(tensors, dim=k)` **inserts a new axis** at position `k`. Output rank = input rank + 1. All inputs must have **identical** shapes.

**Mental model.** If you have a list of `(3, 4)` tensors:
- `t.cat(list, dim=0)` over `N` of them → `(N*3, 4)`
- `t.stack(list, dim=0)` over `N` of them → `(N, 3, 4)`

**Equivalence.** `t.stack(xs, dim=k)` ≡ `t.cat([x.unsqueeze(k) for x in xs], dim=k)`. Same result, but `stack` is shorter and intent-revealing.

### Exercise 1 — pick stack or cat from the target shape

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Choose between `t.stack` and `t.cat` based on whether the target shape introduces a new axis (stack) or extends an existing axis (cat), and dispatch correctly.
> Keywords: stack, cat, rank-change, shape-reasoning
> ```

**KCs targeted:** `stack-inserts-axis`, `cat-along-existing-axis`

Implement `ex1_combine(tensors, target_shape)`. Given:
- `tensors`: a list of N tensors, all the same shape `(d0, d1, ..., dM)`.
- `target_shape`: the desired output shape (a tuple).

Pick `t.stack` or `t.cat` and the correct `dim` based on `target_shape`. Specifically:
- If `len(target_shape) == M + 2` (one axis added — rank increased), use `t.stack(tensors, dim=k)` where `k` is the position of the new axis with size `N`.
- If `len(target_shape) == M + 1` (same rank as inputs — one axis grew by factor `N`), use `t.cat(tensors, dim=k)` where `k` is the axis whose size is `N * d[k]`.
- Otherwise raise `ValueError('cannot combine')`.

Return: the combined tensor of shape `target_shape`.

In [ ]:
def ex1_combine(tensors: list, target_shape: tuple) -> Tensor:
    """Dispatch to stack or cat based on target_shape."""
    raise NotImplementedError()


def _test_ex1():
    # Three (3, 4) tensors --------------------------------------------
    ts = [t.full((3, 4), float(i)) for i in range(3)]

    # Target (3, 3, 4): rank +1, new axis size 3 at position 0 → stack(dim=0).
    out_a = ex1_combine(ts, target_shape=(3, 3, 4))
    assert out_a.shape == (3, 3, 4), f'got {tuple(out_a.shape)}'
    assert t.equal(out_a, t.stack(ts, dim=0))

    # Target (3, 3, 4): same target_shape works → confirms stack picked.
    # Target (3, 4, 3): rank +1, new axis at position 2 → stack(dim=2).
    out_b = ex1_combine(ts, target_shape=(3, 4, 3))
    assert out_b.shape == (3, 4, 3)
    assert t.equal(out_b, t.stack(ts, dim=2))

    # Target (9, 4): same rank, axis 0 grew 3→9 = 3*3 → cat(dim=0).
    out_c = ex1_combine(ts, target_shape=(9, 4))
    assert out_c.shape == (9, 4)
    assert t.equal(out_c, t.cat(ts, dim=0))

    # Target (3, 12): same rank, axis 1 grew 4→12 = 3*4 → cat(dim=1).
    out_d = ex1_combine(ts, target_shape=(3, 12))
    assert out_d.shape == (3, 12)
    assert t.equal(out_d, t.cat(ts, dim=1))

    # Target (2, 3, 4): wrong rank gap (M=2, target rank 3, but new
    # axis would need size 3 not 2) → must raise ValueError.
    try:
        ex1_combine(ts, target_shape=(2, 3, 4))
        raise AssertionError('expected ValueError for incompatible shape')
    except ValueError:
        pass

    # Confirm via dtypes too: combine of 3 long tensors should stay long.
    longs = [t.tensor([1, 2, 3], dtype=t.long) for _ in range(4)]
    out_l = ex1_combine(longs, target_shape=(4, 3))
    assert out_l.dtype == t.long
    assert out_l.shape == (4, 3)
    print('stack/cat dispatched correctly for 5 shape patterns')
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_combine(tensors: list, target_shape: tuple) -> Tensor:
    N = len(tensors)
    in_shape = tuple(tensors[0].shape)
    M = len(in_shape)
    tgt = tuple(target_shape)
    if len(tgt) == M + 1:
        # stack — find the axis whose size is N.
        for k, sz in enumerate(tgt):
            if sz == N and tgt[:k] + tgt[k + 1:] == in_shape:
                return t.stack(tensors, dim=k)
        raise ValueError('cannot combine')
    elif len(tgt) == M:
        # cat — find the axis where target = N * input size.
        for k in range(M):
            if tgt[k] == N * in_shape[k] and all(
                tgt[j] == in_shape[j] for j in range(M) if j != k
            ):
                return t.cat(tensors, dim=k)
        raise ValueError('cannot combine')
    else:
        raise ValueError('cannot combine')
```

**Single-axis-change is the deciding question.** If the output needs a brand-new axis, that's `stack`; if it needs one existing axis to grow, that's `cat`. Anything else is incompatible.

**Why this dispatch matters in real code.** ARENA exercises switch between the two constantly — `stack` for assembling per-head/per-batch results into a leading-axis tensor, `cat` for assembling residual-stream contributions or aggregating logits across model copies. Confusing them produces a tensor that's either 1 rank too high or 1 rank too low — usually caught by a downstream shape-assertion crash.

**The bare-functional rewrite.** `t.stack(xs, dim=k)` is literally `t.cat([x.unsqueeze(k) for x in xs], dim=k)`. Knowing this lets you read source code that uses one when you would've used the other.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()